SAHIE = Small Area Health Insurance Estimates
--> Regular ACS health insurance data has large margins of error for small counties like Tehama.
--> SAHIE uses a statistical model combining multiple data sources to produce more reliable estimates for small areas.

In [1]:
# Imports and Constants 

import requests
import pandas as pd
import time
from pathlib import Path
from dotenv import load_dotenv
import os

load_dotenv()
CENSUS_API_KEY = os.getenv("CENSUS_API_KEY")

# Constants 
TEHAMA_FIPS  = "103"
STATE_FIPS   = "06"

# SAHIE available years
# Note: SAHIE lags ACS by ~2 years
SAHIE_YEARS  = list(range(2010, 2023))  # 2010-2022

# Output folder
RAW_DIR = Path("data/raw/sahie")
RAW_DIR.mkdir(parents=True, exist_ok=True)

print("Setup complete")
print(f"   Years     : {SAHIE_YEARS[0]} → {SAHIE_YEARS[-1]}")
print(f"   Total     : {len(SAHIE_YEARS)} years")
print(f"   Output    : {RAW_DIR}")

Setup complete
   Years     : 2010 → 2022
   Total     : 13 years
   Output    : data\raw\sahie


In [8]:
# SAHIE Fetch Function 

def fetch_sahie(year, geography):
    """
    Fetch SAHIE health insurance data.
    
    Available years: 2006-2023
    Note: SAHIE only covers "Under 65 years" population
          because 65+ are covered by Medicare
    """

    # Build geography parameter 
    if geography == "county":
        geo_param = (f"for=county:{TEHAMA_FIPS}"
                     f"&in=state:{STATE_FIPS}")
        label = "Tehama County"
        fips  = "06103"
    elif geography == "state":
        geo_param = f"for=state:{STATE_FIPS}"
        label = "California"
        fips  = STATE_FIPS
    elif geography == "nation":
        geo_param = "for=us:1"
        label = "United States"
        fips  = "US"

    url = (f"https://api.census.gov/data/timeseries/healthins/sahie"
           f"?get=NAME,NIC_PT,NUI_PT,PCTUI_PT,NIPR_PT"
           f"&{geo_param}"
           f"&AGECAT=0"
           f"&SEXCAT=0"
           f"&IPRCAT=0"
           f"&RACECAT=0"
           f"&time={year}"
           f"&key={CENSUS_API_KEY}")

    resp = requests.get(url, timeout=30)
    resp.raise_for_status()
    data = resp.json()

    #  Parse response 
    row = dict(zip(data[0], data[1]))

    result = {
        "geography"       : label,
        "geo_type"        : geography,
        "sahie_year"      : year,
        "fips"            : fips,
        "total_under_65"  : pd.to_numeric(row.get("NIPR_PT"), errors="coerce"),
        "num_insured"     : pd.to_numeric(row.get("NIC_PT"),  errors="coerce"),
        "num_uninsured"   : pd.to_numeric(row.get("NUI_PT"),  errors="coerce"),
        "pct_uninsured"   : pd.to_numeric(row.get("PCTUI_PT"),errors="coerce"),
        "fetched_at"      : pd.Timestamp.now('UTC').isoformat(),
    }

    # ── Calculate pct insured ─────────────────────────────────
    if pd.notna(result["pct_uninsured"]):
        result["pct_insured"] = round(100 - result["pct_uninsured"], 1)
    else:
        result["pct_insured"] = pd.NA

    return pd.DataFrame([result])

In [10]:
# Fetch All SAHIE Data 

import time

SAHIE_YEARS  = list(range(2010, 2024))  # 2010-2023
GEOGRAPHIES  = ["county", "state", "nation"]

frames = []
errors = []

print("FETCHING ALL SAHIE DATA")
print("=" * 50)
print(f"Years      : {SAHIE_YEARS[0]} → {SAHIE_YEARS[-1]}")
print(f"Geographies: Tehama, California, US")
print(f"Total calls: {len(SAHIE_YEARS) * len(GEOGRAPHIES)}")
print("=" * 50)

for geo in GEOGRAPHIES:
    print(f"\nFetching {geo}...")
    for year in SAHIE_YEARS:
        try:
            df = fetch_sahie(year, geo)
            frames.append(df)
            print(f"{year}")
            time.sleep(0.5)
        except Exception as e:
            print(f"{year} failed: {e}")
            errors.append(f"{geo} {year}: {e}")

# Combine
sahie_df = pd.concat(frames, ignore_index=True)

print("\n" + "=" * 50)
print(f" Fetch complete")
print(f"   Total rows    : {len(sahie_df)}")
print(f"   Tehama rows   : {len(sahie_df[sahie_df['geography']=='Tehama County'])}")
print(f"   CA rows       : {len(sahie_df[sahie_df['geography']=='California'])}")
print(f"   US rows       : {len(sahie_df[sahie_df['geography']=='United States'])}")

if errors:
    print(f"\n{len(errors)} errors:")
    for e in errors:
        print(f"   {e}")

FETCHING ALL SAHIE DATA
Years      : 2010 → 2023
Geographies: Tehama, California, US
Total calls: 42

Fetching county...
2010
2011
2012
2013
2014
2015
2016
2017
2018
2019
2020
2021
2022
2023

Fetching state...
2010
2011
2012
2013
2014
2015
2016
2017
2018
2019
2020
2021
2022
2023

Fetching nation...
2010
2011
2012
2013
2014
2015
2016
2017
2018
2019
2020
2021
2022
2023

 Fetch complete
   Total rows    : 42
   Tehama rows   : 14
   CA rows       : 14
   US rows       : 14


In [12]:
# Save SAHIE Data 

from pathlib import Path

RAW_DIR = Path("data/raw/sahie")
RAW_DIR.mkdir(parents=True, exist_ok=True)

# Save as Parquet 
parquet_path = RAW_DIR / "sahie_raw.parquet"
sahie_df.to_parquet(parquet_path, index=False)

# Save as CSV
csv_path = RAW_DIR / "sahie_raw.csv"
sahie_df.to_csv(csv_path, index=False)

print(f"Parquet saved : {parquet_path}")
print(f"   Size          : {parquet_path.stat().st_size/1024:.1f} KB")
print(f"\nCSV saved     : {csv_path}")
print(f"   Size          : {csv_path.stat().st_size/1024:.1f} KB")

print(f"\nContents:")
print(f"   Rows          : {len(sahie_df)}")
print(f"   Columns       : {len(sahie_df.columns)}")
print(f"   Years         : {sahie_df['sahie_year'].min()} → "
      f"{sahie_df['sahie_year'].max()}")
print(f"   Geographies   : {sahie_df['geography'].unique().tolist()}")

Parquet saved : data\raw\sahie\sahie_raw.parquet
   Size          : 8.1 KB

CSV saved     : data\raw\sahie\sahie_raw.csv
   Size          : 4.1 KB

Contents:
   Rows          : 42
   Columns       : 10
   Years         : 2010 → 2023
   Geographies   : ['Tehama County', 'California', 'United States']


In [16]:
df= pd.read_parquet("data/raw/sahie/sahie_raw.parquet")
df

,geography,geo_type,sahie_year,fips,total_under_65,num_insured,num_uninsured,pct_uninsured,fetched_at,pct_insured
0,Tehama County,county,2010,06103,53162,41741,11421,21.5,2026-04-22T21:52:21.055022+00:00,78.5
1,Tehama County,county,2011,06103,53003,41767,11235,21.2,2026-04-22T21:52:22.763954+00:00,78.8
2,Tehama County,county,2012,06103,52394,41736,10658,20.3,2026-04-22T21:52:24.513091+00:00,79.7
3,Tehama County,county,2013,06103,51680,41707,9973,19.3,2026-04-22T21:52:26.130957+00:00,80.7
4,Tehama County,county,2014,06103,51317,44502,6815,13.3,2026-04-22T21:52:27.723971+00:00,86.7
5,Tehama County,county,2015,06103,51236,46502,4734,9.2,2026-04-22T21:52:29.510459+00:00,90.8
6,Tehama County,county,2016,06103,50753,46832,3921,7.7,2026-04-22T21:52:31.140515+00:00,92.3
7,Tehama County,county,2017,06103,50757,46789,3968,7.8,2026-04-22T21:52:33.729050+00:00,92.2
8,Tehama County,county,2018,06103,50558,46462,4096,8.1,2026-04-22T21:52:35.332228+00:00,91.9
9,Tehama County,county,2019,06103,51138,46690,4448,8.7,2026-04-22T21:52:36.923066+00:00,91.3
